In [ ]:
# ============================================================
# 0. Imports
# ============================================================

import os
import random
import json
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


In [ ]:
# ============================================================
# 1. Reproducibility
# ============================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as e:
            print(e)


In [ ]:
# ============================================================
# 2. Configuration
# ============================================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 6

RUN_STANDARD_CNN = True
RUN_MOBILENETV2 = True
RUN_MOBILEVIT = True

LABEL_SMOOTHING = 0.02

EPOCHS_STANDARD_CNN = 150
EPOCHS_MOBILENET_STAGE1 = 20
EPOCHS_MOBILENET_STAGE2 = 150
EPOCHS_MOBILEVIT = 50

EARLY_STOP_PATIENCE = 20
REDUCE_LR_PATIENCE = 6

EXPECTED_CLASSES = [
    "Algal_leaf_spot",
    "Black_blight",
    "Blister_blight",
    "Gray_blight",
    "Healthy",
    "Spider_mites"
]

SAVE_DIR = Path("/kaggle/working/sltealeaf_three_models_only")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_DIR = SAVE_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_DIR = SAVE_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

FIG_DIR = SAVE_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = SAVE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", SAVE_DIR)


In [ ]:
# ============================================================
# 3. Locate dataset root automatically
# ============================================================

def find_dataset_root(expected_classes=EXPECTED_CLASSES):
    search_roots = [
        Path("/kaggle/input/datasets/vijayakanthan/sltea-raw-aug"),
        Path("/kaggle/input/sltea-raw-aug"),
        Path("/kaggle/input")
    ]

    candidate_dirs = []

    for search_root in search_roots:
        if not search_root.exists():
            continue

        for p in search_root.rglob("*"):
            if p.is_dir():
                try:
                    child_names = {c.name for c in p.iterdir() if c.is_dir()}
                except Exception:
                    continue

                if set(expected_classes).issubset(child_names):
                    candidate_dirs.append(p)

    if not candidate_dirs:
        print("Available /kaggle/input content:")
        input_root = Path("/kaggle/input")
        if input_root.exists():
            for p in input_root.iterdir():
                print(" -", p)
        raise FileNotFoundError(
            "Could not find dataset root containing all class folders. "
            "Check your Kaggle dataset path."
        )

    print("Candidate dataset roots:")
    for c in candidate_dirs:
        print(" -", c)

    return candidate_dirs[0]


DATASET_ROOT = find_dataset_root()
print("Using DATASET_ROOT:", DATASET_ROOT)


In [ ]:
# ============================================================
# 4. Build manifest from folder structure
# ============================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def parse_image_record(image_path, class_name):
    stem = image_path.stem

    if "__aug__" in stem:
        base_id = stem.split("__aug__")[0]
        is_augmented = 1
    else:
        base_id = stem
        is_augmented = 0

    return {
        "image_path": str(image_path),
        "filename": image_path.name,
        "class_label": class_name,
        "base_id": base_id,
        "is_augmented": is_augmented
    }


records = []

for class_name in EXPECTED_CLASSES:
    class_dir = DATASET_ROOT / class_name

    if not class_dir.exists():
        raise FileNotFoundError(f"Missing class folder: {class_dir}")

    for image_path in sorted(class_dir.iterdir()):
        if image_path.suffix.lower() in IMAGE_EXTENSIONS:
            records.append(parse_image_record(image_path, class_name))

df = pd.DataFrame(records)

if df.empty:
    raise ValueError("No images found. Check the dataset folder structure.")

print("Total images:", len(df))
display(df.head())

print("\nImage type counts:")
print(df["is_augmented"].value_counts())

print("\nClass counts:")
print(df["class_label"].value_counts())

print("\nClass counts by original/augmented:")
display(pd.crosstab(df["class_label"], df["is_augmented"]))

print("\nUnique base images:", df["base_id"].nunique())

df.to_csv(SAVE_DIR / "auto_generated_manifest_from_folders.csv", index=False)


In [ ]:
# ============================================================
# 5. Check augmented images have matching original images
# ============================================================

original_base_ids = set(df[df["is_augmented"] == 0]["base_id"])
augmented_base_ids = set(df[df["is_augmented"] == 1]["base_id"])

missing_original_for_aug = augmented_base_ids - original_base_ids

print("Augmented base IDs without original:", len(missing_original_for_aug))

if len(missing_original_for_aug) > 0:
    print("Examples:", list(missing_original_for_aug)[:10])
    raise ValueError(
        "Some augmented images do not have matching original images. "
        "Fix filenames or remove unmatched augmented files."
    )


In [ ]:
# ============================================================
# 6. Split by base_id, not image file
# ============================================================
# Split ratio: 70% train, 15% validation, 15% test.
# Validation and test are original-only.

base_df = (
    df[df["is_augmented"] == 0]
    [["base_id", "class_label"]]
    .drop_duplicates()
    .copy()
)

print("\nOriginal base image count:", len(base_df))
print(base_df["class_label"].value_counts())

train_base_df, temp_base_df = train_test_split(
    base_df,
    test_size=0.30,
    random_state=SEED,
    stratify=base_df["class_label"]
)

val_base_df, test_base_df = train_test_split(
    temp_base_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_base_df["class_label"]
)

train_base_ids = set(train_base_df["base_id"])
val_base_ids = set(val_base_df["base_id"])
test_base_ids = set(test_base_df["base_id"])

assert train_base_ids.isdisjoint(val_base_ids)
assert train_base_ids.isdisjoint(test_base_ids)
assert val_base_ids.isdisjoint(test_base_ids)

print("\nBase-image split:")
print("Train base images:", len(train_base_ids))
print("Validation base images:", len(val_base_ids))
print("Test base images:", len(test_base_ids))


In [ ]:
# ============================================================
# 7. Build train/validation/test dataframes
# ============================================================

# Training uses original + augmented images from train base IDs
train_df = df[df["base_id"].isin(train_base_ids)].copy()

# Validation uses original images only
val_df = df[
    (df["base_id"].isin(val_base_ids)) &
    (df["is_augmented"] == 0)
].copy()

# Test uses original images only
test_df = df[
    (df["base_id"].isin(test_base_ids)) &
    (df["is_augmented"] == 0)
].copy()

# Leakage checks
assert set(train_df["base_id"]).isdisjoint(set(val_df["base_id"]))
assert set(train_df["base_id"]).isdisjoint(set(test_df["base_id"]))
assert set(val_df["base_id"]).isdisjoint(set(test_df["base_id"]))

assert val_df["is_augmented"].sum() == 0
assert test_df["is_augmented"].sum() == 0

print("\nFinal image counts:")
print("Train:", len(train_df), "| original:", (train_df["is_augmented"] == 0).sum(), "| augmented:", train_df["is_augmented"].sum())
print("Validation:", len(val_df), "| original:", (val_df["is_augmented"] == 0).sum(), "| augmented:", val_df["is_augmented"].sum())
print("Test:", len(test_df), "| original:", (test_df["is_augmented"] == 0).sum(), "| augmented:", test_df["is_augmented"].sum())

print("\nTrain class counts:")
print(train_df["class_label"].value_counts())

print("\nValidation class counts:")
print(val_df["class_label"].value_counts())

print("\nTest class counts:")
print(test_df["class_label"].value_counts())

print("\nLeakage check passed: no base_id appears in more than one subset.")
print("Validation and test sets contain original images only.")


In [ ]:
# ============================================================
# 8. Save exact split files for reviewers
# ============================================================

train_df.to_csv(SPLIT_DIR / "train_original_plus_augmented_baseid_split.csv", index=False)
val_df.to_csv(SPLIT_DIR / "validation_original_only_baseid_split.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test_original_only_baseid_split.csv", index=False)

split_summary = pd.DataFrame({
    "subset": ["train", "validation", "test"],
    "base_images": [len(train_base_ids), len(val_base_ids), len(test_base_ids)],
    "total_images": [len(train_df), len(val_df), len(test_df)],
    "original_images": [
        int((train_df["is_augmented"] == 0).sum()),
        int((val_df["is_augmented"] == 0).sum()),
        int((test_df["is_augmented"] == 0).sum())
    ],
    "augmented_images": [
        int(train_df["is_augmented"].sum()),
        int(val_df["is_augmented"].sum()),
        int(test_df["is_augmented"].sum())
    ]
})

split_summary.to_csv(SPLIT_DIR / "split_summary.csv", index=False)
print("\nSplit summary:")
display(split_summary)


In [ ]:
# ============================================================
# 9. Label mapping
# ============================================================

class_names = sorted(EXPECTED_CLASSES)
label_to_id = {label: idx for idx, label in enumerate(class_names)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

print("\nLabel mapping:")
print(label_to_id)

with open(SAVE_DIR / "class_mapping.json", "w") as f:
    json.dump(label_to_id, f, indent=2)


In [ ]:
# ============================================================
# 10. Image loading for Standard CNN and MobileViT
# ============================================================

def load_images_from_dataframe(dataframe, image_size=IMG_SIZE):
    images = []
    labels = []

    for _, row in dataframe.iterrows():
        image_path = row["image_path"]

        img = cv2.imread(image_path)
        if img is None:
            raise FileNotFoundError(f"Could not read image: {image_path}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, image_size, interpolation=cv2.INTER_AREA)

        images.append(img)
        labels.append(label_to_id[row["class_label"]])

    X = np.asarray(images, dtype=np.float32) / 255.0
    y_idx = np.asarray(labels, dtype=np.int64)
    Y = to_categorical(y_idx, num_classes=len(class_names)).astype(np.float32)

    return X, y_idx, Y


print("\nLoading images for Standard CNN and MobileViT...")
X_train, y_train_idx, Y_train = load_images_from_dataframe(train_df)
X_val, y_val_idx, Y_val = load_images_from_dataframe(val_df)
X_test, y_test_idx, Y_test = load_images_from_dataframe(test_df)

print("X_train:", X_train.shape, "Y_train:", Y_train.shape)
print("X_val:", X_val.shape, "Y_val:", Y_val.shape)
print("X_test:", X_test.shape, "Y_test:", Y_test.shape)


In [ ]:
# ============================================================
# 11. Class weights
# ============================================================

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_idx),
    y=y_train_idx
)

class_weight_dict = {
    int(class_id): float(weight)
    for class_id, weight in zip(np.unique(y_train_idx), class_weights_array)
}

print("\nClass weights:")
for class_id, weight in class_weight_dict.items():
    print(id_to_label[class_id], ":", round(weight, 4))

pd.DataFrame({
    "class_id": list(class_weight_dict.keys()),
    "class_label": [id_to_label[i] for i in class_weight_dict.keys()],
    "class_weight": list(class_weight_dict.values())
}).to_csv(REPORT_DIR / "class_weights.csv", index=False)


In [ ]:
# ============================================================
# 12. TensorFlow datasets for Standard CNN and MobileViT
# ============================================================

train_ds = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
train_ds = train_ds.shuffle(len(X_train), seed=SEED, reshuffle_each_iteration=True)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, Y_val))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, Y_test))
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Standard CNN and MobileViT datasets created.")


In [ ]:
# ============================================================
# 12A. MobileNetV2-specific TensorFlow datasets
# ============================================================
# MobileNetV2 uses ImageNet preprocessing.

def load_and_preprocess_image_mobilenetv2(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.image.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])

    image = tf.image.resize(image, IMG_SIZE, method="bilinear")
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)

    label = tf.one_hot(label, depth=len(class_names))
    return image, label


def make_mobilenetv2_dataset(dataframe, shuffle=False):
    paths = dataframe["image_path"].values.astype(str)
    labels = dataframe["class_label"].map(label_to_id).astype(np.int32).values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess_image_mobilenetv2, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds_mobilenet = make_mobilenetv2_dataset(train_df, shuffle=True)
val_ds_mobilenet = make_mobilenetv2_dataset(val_df, shuffle=False)
test_ds_mobilenet = make_mobilenetv2_dataset(test_df, shuffle=False)

print("MobileNetV2-specific datasets created with preprocess_input().")


In [ ]:
# ============================================================
# 13. Standard CNN
# ============================================================

def build_standard_cnn(input_shape=(224, 224, 3), num_classes=6):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(256, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(512, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.40)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs, name="Standard_CNN")

    model.compile(
        optimizer=AdamW(learning_rate=1e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=["accuracy"]
    )

    return model


In [ ]:
# ============================================================
# 14. MobileNetV2
# ============================================================

def build_mobilenetv2(input_shape=(224, 224, 3), num_classes=6):
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet"
    )

    base_model.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs, name="MobileNetV2")

    model.compile(
        optimizer=AdamW(learning_rate=3e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=["accuracy"]
    )

    return model, base_model


In [ ]:
# ============================================================
# 15. MobileViT-v1 style compact baseline
# ============================================================

def conv_bn_act(x, filters, kernel_size=3, strides=1, activation="swish"):
    x = layers.Conv2D(
        filters,
        kernel_size,
        strides=strides,
        padding="same",
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)
    return x


def inverted_residual_block(x, filters, strides=1, expansion=2):
    in_channels = x.shape[-1]
    hidden_dim = int(in_channels * expansion)

    shortcut = x

    x = conv_bn_act(x, hidden_dim, kernel_size=1, strides=1)
    x = layers.DepthwiseConv2D(
        3,
        strides=strides,
        padding="same",
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("swish")(x)

    x = layers.Conv2D(filters, 1, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    if strides == 1 and in_channels == filters:
        x = layers.Add()([shortcut, x])

    return x


def transformer_block(x, num_heads=2, projection_dim=64, mlp_dim=128):
    shortcut = x

    x1 = layers.LayerNormalization(epsilon=1e-6)(x)
    attention_output = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=projection_dim
    )(x1, x1)

    x2 = layers.Add()([attention_output, shortcut])

    x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
    x3 = layers.Dense(mlp_dim, activation="swish")(x3)
    x3 = layers.Dense(x.shape[-1])(x3)

    return layers.Add()([x3, x2])


def mobilevit_block(x, filters, num_heads=2, projection_dim=64):
    local = conv_bn_act(x, filters, kernel_size=3)
    local = conv_bn_act(local, filters, kernel_size=1)

    h = local.shape[1]
    w = local.shape[2]
    c = local.shape[3]

    patches = layers.Reshape((h * w, c))(local)
    patches = transformer_block(
        patches,
        num_heads=num_heads,
        projection_dim=projection_dim,
        mlp_dim=filters * 2
    )

    folded = layers.Reshape((h, w, c))(patches)

    x = layers.Concatenate()([x, folded])
    x = conv_bn_act(x, filters, kernel_size=1)

    return x


def build_mobilevit_v1_style(input_shape=(224, 224, 3), num_classes=6):
    inputs = layers.Input(shape=input_shape)

    x = conv_bn_act(inputs, 16, kernel_size=3, strides=2)

    x = inverted_residual_block(x, 32, strides=1)
    x = inverted_residual_block(x, 48, strides=2)
    x = inverted_residual_block(x, 48, strides=1)

    x = inverted_residual_block(x, 64, strides=2)
    x = mobilevit_block(x, 64, num_heads=2, projection_dim=32)

    x = inverted_residual_block(x, 96, strides=2)
    x = mobilevit_block(x, 96, num_heads=2, projection_dim=48)

    x = conv_bn_act(x, 128, kernel_size=1)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs, name="MobileViT_v1_style")

    model.compile(
        optimizer=AdamW(learning_rate=3e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=["accuracy"]
    )

    return model


In [ ]:
# ============================================================
# 16. Callbacks and evaluation
# ============================================================

def get_callbacks(model_name):
    return [
        EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOP_PATIENCE,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=REDUCE_LR_PATIENCE,
            min_lr=1e-7,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=str(MODEL_DIR / f"{model_name}_best.keras"),
            monitor="val_loss",
            save_best_only=True
        )
    ]


def plot_high_quality_confusion_matrix(
    cm,
    class_names,
    model_name,
    save_dir,
    normalize=False,
    dpi=600
):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    cm = np.asarray(cm)

    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_display = np.divide(
            cm.astype("float"),
            row_sums,
            out=np.zeros_like(cm, dtype=float),
            where=row_sums != 0
        )
        values_to_show = np.round(cm_display * 100, 1)
        value_format = "{:.1f}"
        colorbar_label = "Percentage (%)"
        filename_suffix = "normalized"
    else:
        cm_display = cm
        values_to_show = cm
        value_format = "{:.0f}"
        colorbar_label = "Count"
        filename_suffix = "counts"

    fig, ax = plt.subplots(figsize=(10.5, 8.5))
    im = ax.imshow(cm_display, interpolation="nearest", cmap="Blues")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel(colorbar_label, rotation=270, labelpad=20, fontsize=14)
    cbar.ax.tick_params(labelsize=12)

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))

    ax.set_xticklabels(class_names, fontsize=13, rotation=35, ha="right")
    ax.set_yticklabels(class_names, fontsize=13)

    ax.set_xlabel("Predicted class", fontsize=15, labelpad=12)
    ax.set_ylabel("True class", fontsize=15, labelpad=12)
    ax.set_title(f"{model_name} Confusion Matrix", fontsize=17, pad=18)

    threshold = cm_display.max() / 2.0 if cm_display.max() > 0 else 0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            text = value_format.format(values_to_show[i, j])
            text_color = "white" if cm_display[i, j] > threshold else "black"

            ax.text(
                j, i, text,
                ha="center",
                va="center",
                color=text_color,
                fontsize=14,
                fontweight="bold"
            )

    ax.set_xticks(np.arange(cm.shape[1] + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(cm.shape[0] + 1) - 0.5, minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=2)
    ax.tick_params(which="minor", bottom=False, left=False)

    for spine in ax.spines.values():
        spine.set_linewidth(1.2)

    fig.tight_layout()

    png_path = save_dir / f"{model_name}_confusion_matrix_{filename_suffix}.png"
    pdf_path = save_dir / f"{model_name}_confusion_matrix_{filename_suffix}.pdf"
    svg_path = save_dir / f"{model_name}_confusion_matrix_{filename_suffix}.svg"

    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight")
    plt.show()

    print(f"Saved PNG: {png_path}")
    print(f"Saved PDF: {pdf_path}")
    print(f"Saved SVG: {svg_path}")


def evaluate_model(model, model_name, test_dataset=None, y_true_idx=None):
    if test_dataset is None:
        test_dataset = test_ds

    if y_true_idx is None:
        y_true_idx = y_test_idx

    y_prob = model.predict(test_dataset, verbose=1)
    y_pred = np.argmax(y_prob, axis=1)

    loss, acc = model.evaluate(test_dataset, verbose=0)

    macro_f1 = f1_score(y_true_idx, y_pred, average="macro")
    weighted_f1 = f1_score(y_true_idx, y_pred, average="weighted")

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)
    print("Test loss:", round(loss, 4))
    print("Accuracy:", round(acc, 4))
    print("Macro F1:", round(macro_f1, 4))
    print("Weighted F1:", round(weighted_f1, 4))

    report = classification_report(
        y_true_idx,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(REPORT_DIR / f"{model_name}_classification_report.csv")

    cm = confusion_matrix(y_true_idx, y_pred)
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_df.to_csv(REPORT_DIR / f"{model_name}_confusion_matrix.csv")

    plot_high_quality_confusion_matrix(
        cm=cm,
        class_names=class_names,
        model_name=model_name,
        save_dir=FIG_DIR,
        normalize=False,
        dpi=600
    )

    plot_high_quality_confusion_matrix(
        cm=cm,
        class_names=class_names,
        model_name=model_name,
        save_dir=FIG_DIR,
        normalize=True,
        dpi=600
    )

    pred_df = test_df.copy()
    pred_df["true_label"] = [id_to_label[i] for i in y_true_idx]
    pred_df["predicted_label"] = [id_to_label[i] for i in y_pred]
    pred_df.to_csv(REPORT_DIR / f"{model_name}_test_predictions.csv", index=False)

    return {
        "Model": model_name,
        "Accuracy": float(acc),
        "Macro_F1": float(macro_f1),
        "Weighted_F1": float(weighted_f1),
        "Test_Loss": float(loss)
    }


In [ ]:
# ============================================================
# 17. Train Standard CNN
# ============================================================

results = []

if RUN_STANDARD_CNN:
    cnn = build_standard_cnn(num_classes=len(class_names))
    cnn.summary()

    history_cnn = cnn.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_STANDARD_CNN,
        class_weight=class_weight_dict,
        callbacks=get_callbacks("Standard_CNN"),
        verbose=1
    )

    cnn.save(MODEL_DIR / "Standard_CNN_final.keras")
    cnn_result = evaluate_model(cnn, "Standard_CNN")
    results.append(cnn_result)
else:
    print("Skipping Standard CNN.")


In [ ]:
# ============================================================
# 18. Train MobileNetV2
# ============================================================

if RUN_MOBILENETV2:
    mobilenet, mobilenet_base = build_mobilenetv2(num_classes=len(class_names))
    mobilenet.summary()

    # Stage 1: frozen backbone
    history_mn_stage1 = mobilenet.fit(
        train_ds_mobilenet,
        validation_data=val_ds_mobilenet,
        epochs=EPOCHS_MOBILENET_STAGE1,
        class_weight=class_weight_dict,
        callbacks=get_callbacks("MobileNetV2_stage1"),
        verbose=1
    )

    # Stage 2: fine-tune upper layers
    mobilenet_base.trainable = True

    for layer in mobilenet_base.layers[:-60]:
        layer.trainable = False

    # Keep BatchNorm layers fixed during fine-tuning
    for layer in mobilenet_base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    mobilenet.compile(
        optimizer=AdamW(learning_rate=1e-5),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=["accuracy"]
    )

    history_mn_stage2 = mobilenet.fit(
        train_ds_mobilenet,
        validation_data=val_ds_mobilenet,
        epochs=EPOCHS_MOBILENET_STAGE2,
        class_weight=class_weight_dict,
        callbacks=get_callbacks("MobileNetV2_finetuned"),
        verbose=1
    )

    mobilenet.save(MODEL_DIR / "MobileNetV2_final.keras")
    mobilenet_result = evaluate_model(
        mobilenet,
        "MobileNetV2",
        test_dataset=test_ds_mobilenet,
        y_true_idx=y_test_idx
    )
    results.append(mobilenet_result)
else:
    print("Skipping MobileNetV2.")


In [ ]:
# ============================================================
# 19. Train MobileViT-v1 style model
# ============================================================

if RUN_MOBILEVIT:
    mobilevit = build_mobilevit_v1_style(num_classes=len(class_names))
    mobilevit.summary()

    history_mobilevit = mobilevit.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_MOBILEVIT,
        class_weight=class_weight_dict,
        callbacks=get_callbacks("MobileViT_v1_style"),
        verbose=1
    )

    mobilevit.save(MODEL_DIR / "MobileViT_v1_style_final.keras")
    mobilevit_result = evaluate_model(mobilevit, "MobileViT_v1_style")
    results.append(mobilevit_result)
else:
    print("Skipping MobileViT-v1 style model.")


In [ ]:
# ============================================================
# 20. Save final result summary
# ============================================================

results_df = pd.DataFrame(results)
results_df = results_df[["Model", "Accuracy", "Macro_F1", "Weighted_F1", "Test_Loss"]]

results_df.to_csv(REPORT_DIR / "overall_baseline_results.csv", index=False)

print("\nFinal baseline results:")
display(results_df)


In [ ]:
# ============================================================
# 21. Combined per-class table for manuscript
# ============================================================

report_files = {
    "Standard CNN": REPORT_DIR / "Standard_CNN_classification_report.csv",
    "MobileNetV2": REPORT_DIR / "MobileNetV2_classification_report.csv",
    "MobileViT-v1": REPORT_DIR / "MobileViT_v1_style_classification_report.csv",
}

rows = []

for model_name, report_path in report_files.items():
    if not report_path.exists():
        continue

    rep = pd.read_csv(report_path, index_col=0)
    for cls in class_names:
        if cls in rep.index:
            rows.append({
                "Class": cls,
                "Model": model_name,
                "Precision": rep.loc[cls, "precision"],
                "Recall": rep.loc[cls, "recall"],
                "F1-score": rep.loc[cls, "f1-score"],
                "Support": int(rep.loc[cls, "support"]),
            })

per_class_long = pd.DataFrame(rows)
per_class_long.to_csv(REPORT_DIR / "per_class_results_long.csv", index=False)

if not per_class_long.empty:
    per_class_wide = per_class_long.pivot(index="Class", columns="Model", values=["Precision", "Recall", "F1-score"])
    per_class_wide.to_csv(REPORT_DIR / "per_class_results_wide.csv")
    display(per_class_long)
else:
    print("No per-class table created because no report files were found.")


In [ ]:
# ============================================================
# 22. Training curves
# ============================================================

def save_training_curves(history, model_name):
    if history is None:
        return

    history_df = pd.DataFrame(history.history)
    history_df.to_csv(REPORT_DIR / f"{model_name}_training_history.csv", index=False)

    # Accuracy
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history_df["accuracy"], label="Training accuracy")
    ax.plot(history_df["val_accuracy"], label="Validation accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} Training and Validation Accuracy")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{model_name}_accuracy_curve.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{model_name}_accuracy_curve.pdf", bbox_inches="tight")
    plt.show()

    # Loss
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history_df["loss"], label="Training loss")
    ax.plot(history_df["val_loss"], label="Validation loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(f"{model_name} Training and Validation Loss")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{model_name}_loss_curve.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{model_name}_loss_curve.pdf", bbox_inches="tight")
    plt.show()


if RUN_STANDARD_CNN and "history_cnn" in globals():
    save_training_curves(history_cnn, "Standard_CNN")

if RUN_MOBILENETV2 and "history_mn_stage1" in globals():
    save_training_curves(history_mn_stage1, "MobileNetV2_stage1")

if RUN_MOBILENETV2 and "history_mn_stage2" in globals():
    save_training_curves(history_mn_stage2, "MobileNetV2_finetuned")

if RUN_MOBILEVIT and "history_mobilevit" in globals():
    save_training_curves(history_mobilevit, "MobileViT_v1_style")


In [ ]:
# ============================================================
# 23. Save method description for manuscript/reviewer response
# ============================================================

method_note = f"""
Revised baseline validation protocol

The dataset was split at the base-image level. For each image, the base_id was extracted from the filename before the '__aug__' marker. Original images and all augmented variants derived from the same base image were assigned to the same subset.

Training subset:
- Original and augmented images from the training base IDs.

Validation subset:
- Original standardized images only.
- No augmented images.

Test subset:
- Original standardized images only.
- No augmented images.

No base_id appears in more than one subset. This prevents augmented variants derived from the same original image from appearing across training, validation, and test subsets.

Models retained in this notebook:
- Standard CNN
- MobileNetV2
- MobileViT-v1 style model


Preprocessing:
- Standard CNN and MobileViT-v1 style model used pixel scaling to [0, 1].
- MobileNetV2 used tensorflow.keras.applications.mobilenet_v2.preprocess_input.

Split summary:
{split_summary.to_string(index=False)}

Final baseline results:
{results_df.to_string(index=False)}
"""

with open(SAVE_DIR / "revised_validation_protocol_three_models_only.txt", "w") as f:
    f.write(method_note)

print("\nSaved all outputs to:", SAVE_DIR)
print(method_note)
